# Analyse Bayesienne 

## Ajout des bibliothèques nécessaires

In [18]:
#importation des données
import os
import glob

# Manipulation des données
import pandas as pd
import numpy as np

# Outils de Machine Learning (ici je vais utiliser Scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

print("Toutes les bibliothèques sont chargées avec succès !")

Toutes les bibliothèques sont chargées avec succès !


In [ ]:
def decouper_texte(texte, taille_bloc=100):
    mots = texte.split()
    blocs = []
    # On parcourt la liste de mots en sautant de 'taille_bloc' en 'taille_bloc'
    for i in range(0, len(mots), taille_bloc):
        bloc = " ".join(mots[i:i + taille_bloc])
        if len(bloc.split()) >= (taille_bloc / 2): # On ignore les tout petits bouts à la fin
            blocs.append(bloc)
    return blocs

In [30]:
chemin_dossier = "DATA_AB" 

donnees = []

# On cherche tous les fichiers qui se terminent par .txt dans le dossier
liste_fichiers = glob.glob(os.path.join(chemin_dossier, "*.txt"))

print(f"Trouvé {len(liste_fichiers)} fichiers à analyser.")

for chemin_fichier in liste_fichiers:
    # On isole juste le nom du fichier (ex: "A_Hugo_Miserables.txt")
    nom_fichier = os.path.basename(chemin_fichier)
    
    # 1. Détermination de la classe (La Vérité Terrain)
    # Si le nom commence par "A" (ou "A_"), c'est la classe 0 (Non-Naturaliste)
    if nom_fichier.endswith("A.txt"):
        label = "non_naturaliste" # Classe 0
    else:
        label = "naturaliste" # Sinon, c'est du Zola/Goncourt (Classe 1)
        
    # 2. Lecture du fichier
    # Note : 'utf-8' est le standard, mais si vous avez des erreurs de caractères, 
    # essayez encoding='latin-1' (très fréquent sur les vieux textes français)
    with open(chemin_fichier, 'r', encoding='utf-8') as f:
        texte_entier = f.read()
        
    # 3. Découpage du roman en petits blocs de 1000 mots
    blocs = decouper_texte(texte_entier)
    
    # 4. Ajout de chaque extrait dans notre base de données
    for bloc in blocs:
        donnees.append({
            "texte": bloc, 
            "label": label, 
            "source": nom_fichier # Pratique pour vérifier d'où vient une erreur du modèle !
        })

# Conversion finale en tableau Pandas
df = pd.DataFrame(donnees)

print("-" * 30)
print(f"Le modèle a maintenant {len(df)} extraits pour s'entraîner.")
print(df['label'].value_counts()) # Affiche l'équilibre entre vos deux classes
display(df.head())

Trouvé 35 fichiers à analyser.
------------------------------
Le modèle a maintenant 30603 extraits pour s'entraîner.
label
naturaliste        15892
non_naturaliste    14711
Name: count, dtype: int64


,texte,label,source
0,PREMIÈRE PARTIE Chapitre I. Il y avait bien tr...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
1,"mois d'août.Certes, il avait été très beau le ...",non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
2,sa parole... Enfin tout avait donc été dit ! L...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
3,délicate. Bien des grands yeux ardents se fixa...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
4,arrêtée devant la maison d'école. On entendait...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt


In [31]:
df


,texte,label,source
0,PREMIÈRE PARTIE Chapitre I. Il y avait bien tr...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
1,"mois d'août.Certes, il avait été très beau le ...",non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
2,sa parole... Enfin tout avait donc été dit ! L...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
3,délicate. Bien des grands yeux ardents se fixa...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
4,arrêtée devant la maison d'école. On entendait...,non_naturaliste,1883_Lesueur_L'amant-de-Genevieve_A.txt
...,...,...,...
30598,en matière de nouvelles! dit Asraël. Je n'ai l...,non_naturaliste,Berthoud_LHommeDepuisCinqMilleAns_1865_A.txt
30599,indispensable. Le *Journal* annonce que désorm...,non_naturaliste,Berthoud_LHommeDepuisCinqMilleAns_1865_A.txt
30600,contenait les ouvrages les plus célèbres et de...,non_naturaliste,Berthoud_LHommeDepuisCinqMilleAns_1865_A.txt
30601,je vis mon ami et mon médecin le docteur Amédé...,non_naturaliste,Berthoud_LHommeDepuisCinqMilleAns_1865_A.txt


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    df['texte'], 
    df['label'], 
    test_size=0.2, 
    random_state=42 # Assure la reproductibilité
)

# 2. Création du pipeline
modele_bayes = make_pipeline(
    TfidfVectorizer(max_features=5000, lowercase=True),
    MultinomialNB(alpha=1.0) # alpha=1.0 est le lissage de Laplace
)

# 3. Entraînement du modèle
modele_bayes.fit(X_train, y_train)
print("Modèle entraîné avec succès !")

Modèle entraîné avec succès !


In [33]:
# Prédictions sur le corpus de test
predictions = modele_bayes.predict(X_test)

# Affichage du rapport
print("Rapport de classification :")
print(classification_report(y_test, predictions, target_names=["Non-Naturaliste", "Naturaliste"]))

Rapport de classification :
                 precision    recall  f1-score   support

Non-Naturaliste       0.84      0.94      0.89      3159
    Naturaliste       0.93      0.81      0.86      2962

       accuracy                           0.88      6121
      macro avg       0.88      0.87      0.88      6121
   weighted avg       0.88      0.88      0.88      6121

